# Clase 6: Optimización de Modelos de Machine Learning

## Contenido

| # | Tema | Descripción |
|---|------|-------------|
| 01 | **Parámetros vs Hiperparámetros** | Entender qué controla el modelo y qué controlas tú como científico de datos |
| 02 | **Búsqueda de hiperparámetros y Validación Cruzada** | Búsqueda exhaustiva de hiperparámetros con validación cruzada robusta |
| 03 | **Pipeline de sklearn** | Encadenar preprocesamiento y modelo en un solo objeto profesional |
| 04 | **Laboratorio: Optimizar tu Modelo** | Aplicar ajuste de hiperparámetros a tu caso empresarial y medir la mejora obtenida |

---

## Glosario de términos clave

Antes de empezar, repasemos los conceptos que vamos a usar en esta clase:

| Término en inglés | Significado en español | Explicación sencilla |
|---------|----------------------|---------------------|
| **Tuning** | Afinación / Ajuste | Proceso de buscar los mejores valores de configuración para un modelo |
| **Hiperparámetro** | Configuración previa | Un "botón" que tú ajustas ANTES de entrenar el modelo. Ejemplo: cuántos árboles usar |
| **Cross Validation** | Validación cruzada | Técnica para evaluar un modelo de forma más confiable, dividiendo los datos en varias partes |
| **Fold** | Pliegue / Partición | Cada una de las partes en que se dividen los datos durante la validación cruzada |
| **Grid Search** | Búsqueda en cuadrícula | Probar TODAS las combinaciones posibles de hiperparámetros para encontrar la mejor |
| **Pipeline** | Tubería / Flujo de trabajo | Secuencia de pasos (limpiar datos → escalar → entrenar modelo) empaquetados en un solo objeto |
| **Scoring** | Puntuación / Métrica | La medida que usamos para decidir qué tan bueno es un modelo (accuracy, f1-score, etc.) |
| **Estimator** | Estimador | Cualquier objeto de sklearn que puede aprender de datos (un modelo, un escalador, etc.) |
| **n_jobs** | Núcleos de CPU | Cuántos procesadores usar en paralelo. `-1` significa "usar todos los disponibles" |
| **Data Leakage** | Fuga de datos | Error donde el modelo "ve" información del test durante el entrenamiento, dando resultados falsos |
| **StandardScaler** | Escalador estándar | Transforma los datos para que tengan media 0 y desviación estándar 1 |
| **Overfitting** | Sobreajuste | Cuando el modelo memoriza los datos de entrenamiento y no generaliza bien con datos nuevos |

---

## 01. Parámetros vs Hiperparámetros

### ¿Cuál es la diferencia?

| Concepto | Parámetros | Hiperparámetros |
|----------|-----------|-----------------|
| **¿Quién los define?** | El modelo los aprende durante el entrenamiento | El científico de datos los define antes de entrenar |
| **Ejemplo en Regresión Lineal** | Coeficientes (pendiente, intercepto) | No tiene muchos |
| **Ejemplo en Random Forest** | Reglas de cada árbol | `n_estimators`, `max_depth`, `min_samples_split` |
| **Ejemplo en SVM** | Vectores de soporte | `C`, `kernel`, `gamma` |
| **¿Se pueden cambiar manualmente?** | No, se calculan automáticamente | Sí, tú los ajustas |

> **Analogía:** Si un modelo es un carro, los **parámetros** son las piezas internas del motor (las calcula el fabricante). Los **hiperparámetros** son los ajustes que tú haces: tipo de gasolina, presión de llantas, modo de conducción.

### Ejemplo práctico: hiperparámetros de Random Forest

¿Qué significa cada hiperparámetro?

| Hiperparámetro | ¿Qué controla? | Analogía |
|----------------|----------------|----------|
| `n_estimators` | Cantidad de árboles en el bosque | Más árboles = más opiniones para votar. Más es mejor, pero más lento |
| `max_depth` | Profundidad máxima de cada árbol | Qué tan "detalladas" son las preguntas. Muy profundo = memoriza (sobreajuste) |
| `min_samples_split` | Mínimo de datos para dividir un nodo | "Solo hago otra pregunta si tengo al menos X datos". Evita decisiones con poca información |
| `min_samples_leaf` | Mínimo de datos en una hoja final | "Mi respuesta final debe basarse en al menos X datos". Mayor = más conservador |

> **Ejemplo con `max_depth`:** Un árbol con `max_depth=2` hace máximo 2 preguntas (ej: "¿tenure > 12?" → "¿MonthlyCharges > 70?" → respuesta). Un árbol con `max_depth=None` sigue preguntando hasta clasificar perfectamente cada dato (riesgo de sobreajuste).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Cargar dataset local
df = pd.read_csv('Telco-Customer-Churn.csv')

# Preprocesamiento básico
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Seleccionar features numéricas
features = ['tenure', 'MonthlyCharges', 'TotalCharges']
X = df[features]
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape[0]} registros")
print(f"Datos de prueba: {X_test.shape[0]} registros")

### Modelo SIN ajuste de hiperparámetros (valores por defecto)

In [ ]:
# Modelo con hiperparámetros por defecto (sin ajuste)
rf_default = RandomForestClassifier(random_state=42)
rf_default.fit(X_train, y_train)

y_pred_default = rf_default.predict(X_test)
acc_default = accuracy_score(y_test, y_pred_default)

print("=== Random Forest SIN ajuste de hiperparámetros ===")
print(f"Accuracy (precisión general): {acc_default:.4f}")
print(f"\nHiperparámetros usados por defecto:")
print(f"  n_estimators (cantidad de árboles)       = {rf_default.n_estimators}")
print(f"  max_depth (profundidad máxima)            = {rf_default.max_depth}")
print(f"  min_samples_split (mínimo para dividir)   = {rf_default.min_samples_split}")
print(f"  min_samples_leaf (mínimo en hoja final)   = {rf_default.min_samples_leaf}")
print(f"\n{classification_report(y_test, y_pred_default, target_names=['No Churn', 'Churn'])}")

---

## 02. Búsqueda de hiperparámetros y Validación Cruzada

### ¿Qué es la Validación Cruzada (Cross Validation)?

**El problema:** Cuando dividimos datos en entrenamiento/prueba una sola vez, el resultado depende de **cuáles datos cayeron en cada grupo**. Si tuvimos suerte, el accuracy es alto; si no, es bajo. No es confiable.

**La solución:** En vez de dividir una sola vez, **dividimos los datos en K partes iguales llamadas "folds"** (particiones). Luego entrenamos K veces, y cada vez una partición diferente es la de prueba.

### ¿Qué es un Fold (partición)?

Un **fold** (partición) es simplemente una porción de los datos. Si usamos `cv=5` (5 particiones de validación cruzada), dividimos los datos en 5 partes iguales:

```
Imaginemos que tenemos 1000 datos y los dividimos en 5 particiones de 200 cada una:

Ronda 1: [PRUEBA 200] [Entrena 200] [Entrena 200] [Entrena 200] [Entrena 200]  → Accuracy: 0.78
Ronda 2: [Entrena 200] [PRUEBA 200] [Entrena 200] [Entrena 200] [Entrena 200]  → Accuracy: 0.81
Ronda 3: [Entrena 200] [Entrena 200] [PRUEBA 200] [Entrena 200] [Entrena 200]  → Accuracy: 0.79
Ronda 4: [Entrena 200] [Entrena 200] [Entrena 200] [PRUEBA 200] [Entrena 200]  → Accuracy: 0.80
Ronda 5: [Entrena 200] [Entrena 200] [Entrena 200] [Entrena 200] [PRUEBA 200]  → Accuracy: 0.77

Resultado final = Promedio de las 5 rondas = 0.79 ± 0.01
```

> **Ventaja:** El resultado es mucho más confiable porque **todos los datos fueron usados tanto para entrenar como para evaluar**, solo que en rondas diferentes. El "±" nos dice qué tan estable es el modelo.

### ¿Qué es GridSearchCV (búsqueda en cuadrícula con validación cruzada)?

El nombre viene de: **Grid** (cuadrícula) + **Search** (búsqueda) + **CV** (validación cruzada).

Imagina que quieres probar 3 valores de `n_estimators` y 4 valores de `max_depth`. Eso genera una "cuadrícula" de 3×4 = 12 combinaciones. **GridSearchCV prueba TODAS las combinaciones** y evalúa cada una con validación cruzada para encontrar la mejor.

Es como probar todas las combinaciones de ingredientes en una receta hasta encontrar la que sabe mejor.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Definir la grilla de hiperparámetros a probar
param_grid = {
    'n_estimators': [50, 100, 200],        # Cantidad de árboles
    'max_depth': [3, 5, 10, None],          # Profundidad máxima
    'min_samples_split': [2, 5, 10],        # Mínimo de muestras para dividir un nodo
    'min_samples_leaf': [1, 2, 4]           # Mínimo de muestras en una hoja
}

# Calcular total de combinaciones
total = 1
for v in param_grid.values():
    total *= len(v)
print(f"Total de combinaciones a probar: {total}")
print(f"Con 5 folds de validación cruzada: {total * 5} modelos a entrenar")
print(f"(Cada combinación se evalúa 5 veces, una por cada fold)")

In [ ]:
# Ejecutar la búsqueda en cuadrícula con validación cruzada (GridSearchCV)
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,                  # 5 particiones de validación cruzada
    scoring='accuracy',    # Métrica a optimizar (precisión general)
    n_jobs=-1,             # Usar todos los núcleos del procesador
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nMejores hiperparámetros encontrados:")
for param, value in grid_search.best_params_.items():
    print(f"  {param} = {value}")
print(f"\nMejor accuracy (precisión) en validación cruzada: {grid_search.best_score_:.4f}")

### Comparar: modelo por defecto vs modelo optimizado (ajustado)

In [ ]:
# Evaluar el mejor modelo en datos de PRUEBA
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
acc_tuned = accuracy_score(y_test, y_pred_tuned)

print("=" * 50)
print("COMPARACIÓN DE RESULTADOS")
print("=" * 50)
print(f"Accuracy SIN ajuste:  {acc_default:.4f}")
print(f"Accuracy CON ajuste:  {acc_tuned:.4f}")
print(f"Mejora:               {(acc_tuned - acc_default) * 100:.2f} puntos porcentuales")
print("=" * 50)
print(f"\n{classification_report(y_test, y_pred_tuned, target_names=['No Churn', 'Churn'])}")

### Visualizar los resultados de la búsqueda en cuadrícula

In [ ]:
import matplotlib.pyplot as plt

# Resultados de la búsqueda en cuadrícula
results = pd.DataFrame(grid_search.cv_results_)
results = results.sort_values('rank_test_score')

# Top 10 combinaciones
top10 = results.head(10)[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
print("Top 10 combinaciones de hiperparámetros:\n")
for i, row in top10.iterrows():
    print(f"  #{int(row['rank_test_score']):2d} | Accuracy: {row['mean_test_score']:.4f} ± {row['std_test_score']:.4f} | {row['params']}")

# Gráfico: Accuracy vs max_depth para diferentes n_estimators
fig, ax = plt.subplots(figsize=(10, 5))
for n_est in param_grid['n_estimators']:
    mask = results['param_n_estimators'] == n_est
    subset = results[mask].groupby('param_max_depth')['mean_test_score'].max()
    ax.plot(subset.index.astype(str), subset.values, marker='o', label=f'n_estimators={n_est}')

ax.set_xlabel('Profundidad máxima (max_depth)')
ax.set_ylabel('Accuracy (precisión) en validación cruzada')
ax.set_title('Impacto de la profundidad y cantidad de árboles en la precisión')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 03. Pipeline (flujo de trabajo) de sklearn

### ¿Qué es un Pipeline (flujo de trabajo)?

Un **Pipeline** (flujo de trabajo) es una forma de encadenar varios pasos en un solo objeto. En vez de ejecutar cada paso por separado, el Pipeline los ejecuta en orden automáticamente.

```
Sin Pipeline (manual, propenso a errores):
  Paso 1: Escalar datos de entrenamiento
  Paso 2: Escalar datos de prueba (con los mismos valores del paso 1, ¡cuidado!)
  Paso 3: Entrenar modelo con datos escalados
  Paso 4: Predecir con datos de prueba escalados

Con Pipeline (automático):
  pipeline.fit(X_train)    → Escala + Entrena en un solo paso
  pipeline.predict(X_test) → Escala + Predice en un solo paso
```

### ¿Qué es la fuga de datos (Data Leakage)?

Es un error común donde el modelo **"hace trampa"** sin querer. Ejemplo:

- Si escalas TODOS los datos (entrenamiento + prueba) juntos con `StandardScaler` (escalador estándar), el escalador aprende estadísticas de los datos de prueba.
- Entonces el modelo ya "sabe" algo de la prueba antes de evaluarse, y el accuracy es artificialmente alto.
- **El Pipeline evita esto** porque el escalador solo aprende de los datos de entrenamiento.

### ¿Por qué usar Pipelines (flujos de trabajo)?

1. **Código más limpio** — todo en un solo flujo
2. **Evita fuga de datos** — el escalador se ajusta SOLO con datos de entrenamiento
3. **Compatible con GridSearchCV** — se puede hacer ajuste de hiperparámetros de todo el flujo junto
4. **Listo para producción** — un solo objeto para guardar y desplegar

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Sin Pipeline (propenso a errores)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
# model = RandomForestClassifier()
# model.fit(X_train_scaled, y_train)
# y_pred = model.predict(X_test_scaled)

# Con Pipeline (profesional)
pipeline = Pipeline([
    ('scaler', StandardScaler()),              # Paso 1: Escalar (normalizar) datos
    ('classifier', RandomForestClassifier(random_state=42))  # Paso 2: Modelo
])

pipeline.fit(X_train, y_train)
y_pred_pipe = pipeline.predict(X_test)
acc_pipe = accuracy_score(y_test, y_pred_pipe)

print(f"Accuracy (precisión) con Pipeline: {acc_pipe:.4f}")
print(f"\nPasos del flujo de trabajo (pipeline):")
for name, step in pipeline.steps:
    print(f"  {name}: {step.__class__.__name__}")

### Búsqueda en cuadrícula + Pipeline (lo mejor de ambos mundos)

Cuando usamos GridSearchCV (búsqueda en cuadrícula con validación cruzada) con un Pipeline, los nombres de los hiperparámetros se escriben con doble guion bajo: `nombre_del_paso__hiperparámetro`.

```python
# El paso se llama 'classifier', el hiperparámetro es 'max_depth'
# Entonces se escribe: 'classifier__max_depth'

Pipeline([
    ('scaler', StandardScaler()),        # paso llamado "scaler"
    ('classifier', RandomForest()),      # paso llamado "classifier"
])

# Para hacer ajuste del max_depth del clasificador:
param_grid = {'classifier__max_depth': [3, 5, 10]}
#              ^^^^^^^^^^  ^^^^^^^^^
#              nombre paso  hiperparámetro
```

In [ ]:
# Búsqueda en cuadrícula sobre el Pipeline completo
pipe_param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 10],
    'classifier__min_samples_split': [2, 5],
}

grid_pipe = GridSearchCV(
    estimator=pipeline,
    param_grid=pipe_param_grid,
    cv=5,                  # 5 particiones de validación cruzada
    scoring='accuracy',    # Métrica: precisión general
    n_jobs=-1,             # Usar todos los procesadores
    verbose=1
)

grid_pipe.fit(X_train, y_train)

print(f"\nMejores hiperparámetros del Pipeline (flujo de trabajo):")
for param, value in grid_pipe.best_params_.items():
    print(f"  {param} = {value}")

y_pred_best_pipe = grid_pipe.predict(X_test)
acc_best_pipe = accuracy_score(y_test, y_pred_best_pipe)

print(f"\nAccuracy (precisión) en datos de prueba: {acc_best_pipe:.4f}")

---

## 04. Laboratorio: Optimizar tu Modelo

### Instrucciones

Aplica lo aprendido a tu caso empresarial (el dataset que vienes trabajando). Sigue estos pasos:

1. **Carga tu dataset** y prepara X, y
2. **Entrena un modelo base** (sin ajuste de hiperparámetros) y registra su accuracy/f1-score
3. **Define una grilla (cuadrícula)** de hiperparámetros relevantes para tu modelo
4. **Ejecuta GridSearchCV** (búsqueda en cuadrícula con validación cruzada) con cv=5
5. **Compara** el modelo base vs el modelo optimizado (ajustado)
6. **Responde:**
   - ¿Cuánto mejoró tu modelo?
   - ¿Qué hiperparámetro tuvo más impacto?
   - ¿Valió la pena el tiempo de cómputo adicional?

### Referencia rápida de hiperparámetros por modelo

| Modelo | Hiperparámetros clave |
|--------|----------------------|
| **Random Forest** | `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf` |
| **Regresión Logística** | `C`, `penalty`, `solver`, `max_iter` |
| **SVM** | `C`, `kernel`, `gamma` |
| **KNN** | `n_neighbors`, `weights`, `metric` |
| **Naive Bayes** | `var_smoothing` (GaussianNB) |
| **Gradient Boosting** | `n_estimators`, `learning_rate`, `max_depth`, `subsample` |

In [ ]:
# Tu código aquí — Laboratorio
# 1. Carga tu dataset
# 2. Entrena modelo base
# 3. Define param_grid
# 4. Ejecuta GridSearchCV
# 5. Compara resultados